In [0]:
%run "/Workspace/utilities"

In [0]:
dbutils.widgets.text("load_date", "")
load_date = dbutils.widgets.get("load_date")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Configuration

# COMMAND ----------

# Use config from utilities
logger = DataLogger("customer_metrics")

# Paths
customers_path = config.get_silver_path("customers")
orders_path = config.get_silver_path("orders")
gold_path = config.get_gold_path("customer_metrics")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Read Silver Data

# COMMAND ----------

df_customers = spark.read.format("delta").load(customers_path)
df_orders = spark.read.format("delta").load(orders_path) \
    .filter(col("order_status") == "COMPLETED")  # Only completed orders

logger.info(f"Loaded {df_customers.count()} customers and {df_orders.count()} completed orders")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Calculate RFM Metrics

# COMMAND ----------

from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

# Reference date for recency calculation
reference_date = datetime.now()

# Calculate RFM metrics per customer
df_rfm = df_orders.groupBy("customer_id").agg(
    # Recency: Days since last order
    datediff(lit(reference_date), max("order_date")).alias("recency_days"),
    
    # Frequency: Number of orders
    count("order_id").alias("frequency"),
    
    # Monetary: Total spend
    sum("total_amount").alias("monetary"),
    
    # Additional metrics
    avg("total_amount").alias("avg_order_value"),
    min("order_date").alias("first_order_date"),
    max("order_date").alias("last_order_date")
)

# Calculate customer tenure
df_rfm = df_rfm.withColumn(
    "customer_tenure_days",
    datediff(col("last_order_date"), col("first_order_date"))
)

logger.info("RFM metrics calculated")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Calculate RFM Scores (1-5 scale)

# COMMAND ----------

from pyspark.sql.window import Window

# Create quintiles for each RFM dimension
recency_window = Window.orderBy(col("recency_days"))
frequency_window = Window.orderBy(col("frequency").desc())
monetary_window = Window.orderBy(col("monetary").desc())

df_rfm_scored = df_rfm \
    .withColumn("recency_score", 
                ntile(5).over(recency_window)) \
    .withColumn("frequency_score", 
                ntile(5).over(frequency_window)) \
    .withColumn("monetary_score", 
                ntile(5).over(monetary_window))

# Calculate combined RFM score
df_rfm_scored = df_rfm_scored.withColumn(
    "rfm_score",
    (col("recency_score") + col("frequency_score") + col("monetary_score")) / 3
)

logger.info("RFM scores calculated")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Calculate Additional Customer Metrics

# COMMAND ----------

# Calculate purchase patterns
df_purchase_patterns = df_orders.groupBy("customer_id").agg(
    # Order frequency patterns
    countDistinct("order_year").alias("active_years"),
    countDistinct("order_month").alias("active_months"),
    
    # Order value patterns
    min("total_amount").alias("min_order_value"),
    max("total_amount").alias("max_order_value"),
    stddev("total_amount").alias("order_value_std")
)

# Join RFM with purchase patterns
df_metrics = df_rfm_scored.join(df_purchase_patterns, "customer_id", "left")

logger.info("Purchase patterns calculated")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Calculate Customer Lifetime Value (CLV)

# COMMAND ----------

# Simple CLV = Average Order Value × Purchase Frequency × Customer Lifespan
# More sophisticated models can be added later

df_metrics = df_metrics.withColumn(
    "estimated_clv",
    (col("avg_order_value") * col("frequency") * 
     (col("customer_tenure_days") / 365.0))
)

# Categorize CLV
df_metrics = df_metrics.withColumn(
    "clv_category",
    when(col("estimated_clv") < 100, "Low")
    .when((col("estimated_clv") >= 100) & (col("estimated_clv") < 500), "Medium")
    .when((col("estimated_clv") >= 500) & (col("estimated_clv") < 1000), "High")
    .otherwise("Premium")
)

logger.info("CLV calculated")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Enrich with Customer Master Data

# COMMAND ----------

# Join with customer master data
df_final = df_customers.select(
    "customer_id", "full_name", "email", "country", "email_domain"
).join(df_metrics, "customer_id", "inner")

# Add calculated date fields
df_final = df_final \
    .withColumn("metrics_calculated_date", lit(datetime.now())) \
    .withColumn("is_active", 
                when(col("recency_days") <= 90, True).otherwise(False))

logger.info(f"Final customer metrics: {df_final.count()} customers")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Write to Gold Layer

# COMMAND ----------

try:
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("country") \
        .save(gold_path)
    
    logger.info(f"✅ Successfully wrote {df_final.count()} customer metrics to Gold layer")
    
except Exception as e:
    logger.error("Failed to write to Gold layer", e)
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Display Summary Statistics

# COMMAND ----------

print("=== Customer Metrics Summary ===\n")

# Overall stats
overall = df_final.agg(
    count("*").alias("total_customers"),
    sum(col("is_active").cast("int")).alias("active_customers"),
    avg("recency_days").alias("avg_recency"),
    avg("frequency").alias("avg_frequency"),
    avg("monetary").alias("avg_monetary"),
    sum("monetary").alias("total_revenue")
).collect()[0]

print(f"Total Customers: {overall['total_customers']:,}")
print(f"Active Customers: {overall['active_customers']:,}")
print(f"Average Recency: {overall['avg_recency']:.1f} days")
print(f"Average Frequency: {overall['avg_frequency']:.1f} orders")
print(f"Average Monetary: ${overall['avg_monetary']:,.2f}")
print(f"Total Revenue: ${overall['total_revenue']:,.2f}")

# COMMAND ----------

# CLV Distribution
print("\n=== CLV Distribution ===")
clv_dist = df_final.groupBy("clv_category").agg(
    count("*").alias("customer_count"),
    sum("estimated_clv").alias("total_clv")
).orderBy("customer_count", ascending=False)

display(clv_dist)

# COMMAND ----------

# RFM Score Distribution
print("\n=== Top 10 Customers by RFM Score ===")
top_customers = df_final.select(
    "customer_id", "full_name", "email",
    "recency_score", "frequency_score", "monetary_score", "rfm_score",
    "estimated_clv"
).orderBy("rfm_score", ascending=False).limit(10)

display(top_customers)

# COMMAND ----------

logger.info("Customer metrics generation completed successfully")
